# 04 - Benchmark Evaluation

This notebook evaluates the full MLShield cascade (Layer 1 + Layer 2 + Layer 3 fallback) on the benchmark dataset. We measure detection accuracy, cascade efficiency, and temporal security metrics.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import asyncio
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from datetime import datetime, timezone

from mlshield.ingestion.event_bus import TrajectoryEvent, EventSource
from mlshield.specs.spec_validator import SpecValidator
from mlshield.detectors.layer1_rules import RuleEngine
from mlshield.detectors.layer2_ml import MLDetector
from mlshield.detectors.layer3_llm import LLMJudge
from mlshield.detectors.cascade import CascadedDetector
from mlshield.metrics.temporal import TemporalMetrics

## 1. Initialize the Cascade

In [ ]:
validator = SpecValidator(spec_path='../configs/default_specs.yaml')
ml_detector = MLDetector(
    lstm_model_path='../benchmark/data/models/lstm_detector.pt',
    isolation_model_path='../benchmark/data/models/isolation_forest.pkl',
)
llm_judge = LLMJudge()  # No API key -> uses fallback heuristic

cascade = CascadedDetector(
    spec_validator=validator,
    ml_detector=ml_detector,
    llm_judge=llm_judge,
    layer2_threshold=0.6,
    layer3_threshold=0.8,
)

print(f'LSTM loaded: {ml_detector._lstm_loaded}')
print(f'Isolation Forest loaded: {ml_detector.isolation_forest.is_fitted}')
print('Cascade initialized (LLM in fallback mode)')

## 2. Load Benchmark Dataset

In [ ]:
with open('../benchmark/data/mlshield_benchmark_v1.json') as f:
    dataset = json.load(f)

print(f'Total trajectories: {len(dataset)}')
label_counts = Counter(t['label'] for t in dataset)
for label, count in sorted(label_counts.items()):
    print(f'  {label}: {count}')

## 3. Run Full Cascade Evaluation

Process a subset of trajectories through the full cascade and collect results.

In [ ]:
async def evaluate_trajectories(dataset, max_trajs=300):
    results = []
    subset = dataset[:max_trajs]
    
    for i, traj in enumerate(subset):
        is_attack = traj['label'] != 'benign'
        attack_start = traj.get('attack_start_step', None)
        
        if attack_start is not None:
            cascade.temporal_metrics.record_ground_truth_violation(traj['job_id'], attack_start)
        
        traj_threats = []
        for event_data in traj['events']:
            event = TrajectoryEvent(
                event_id=f'{traj["job_id"]}-{event_data.get("step", 0)}',
                timestamp=datetime.now(timezone.utc),
                source=EventSource.K8S_AUDIT,
                job_id=traj['job_id'],
                user='training-user',
                action=event_data.get('action', ''),
                resource=event_data.get('resource', ''),
                details=event_data.get('details', {}),
                trajectory_step=event_data.get('step', 0),
            )
            result = await cascade.evaluate(event)
            if result.is_threat:
                traj_threats.append(result)
        
        detected = len(traj_threats) > 0
        results.append({
            'job_id': traj['job_id'],
            'label': traj['label'],
            'is_attack': is_attack,
            'detected': detected,
            'num_alerts': len(traj_threats),
            'first_detection_layer': traj_threats[0].detected_by_layer if traj_threats else None,
            'first_detection_step': traj_threats[0].step_detected if traj_threats else None,
        })
        
        # Clear LSTM buffer after each trajectory
        ml_detector.clear_buffer(traj['job_id'])
        
        if (i + 1) % 50 == 0:
            print(f'  Processed {i + 1}/{len(subset)} trajectories...')
    
    return pd.DataFrame(results)

results_df = await evaluate_trajectories(dataset, max_trajs=300)
print(f'\nEvaluation complete: {len(results_df)} trajectories')

## 4. Detection Accuracy

In [ ]:
tp = ((results_df['is_attack']) & (results_df['detected'])).sum()
fp = ((~results_df['is_attack']) & (results_df['detected'])).sum()
fn = ((results_df['is_attack']) & (~results_df['detected'])).sum()
tn = ((~results_df['is_attack']) & (~results_df['detected'])).sum()

precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-10)

print('=== Cascade Detection Accuracy ===')
print(f'True Positives:  {tp}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')
print(f'True Negatives:  {tn}')
print(f'\nPrecision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')

## 5. Detection by Attack Type

In [ ]:
attack_results = results_df[results_df['is_attack']]
by_type = attack_results.groupby('label').agg(
    total=('detected', 'count'),
    detected=('detected', 'sum'),
    avg_alerts=('num_alerts', 'mean'),
)
by_type['detection_rate'] = (by_type['detected'] / by_type['total'] * 100).round(1)

print('=== Detection Rate by Attack Type ===')
print(by_type.to_string())

## 6. Cascade Efficiency

In [ ]:
stats = cascade.get_cascade_stats()
print('=== Cascade Efficiency ===')
print(f'Total events processed: {stats["total_events"]:,}')
print(f'Layer 1 cleared:        {stats["layer1_cleared_pct"]:.1f}%')
print(f'Layer 2 processed:      {stats["layer2_processed_pct"]:.1f}%')
print(f'Layer 3 processed:      {stats["layer3_processed_pct"]:.1f}%')

# Detection layer distribution for threats
detected = results_df[results_df['detected']]
print(f'\nFirst detection layer distribution:')
if len(detected) > 0:
    layer_dist = detected['first_detection_layer'].value_counts().sort_index()
    for layer, count in layer_dist.items():
        print(f'  Layer {layer}: {count} ({count/len(detected)*100:.1f}%)')

## 7. Temporal Security Metrics

In [ ]:
temporal = cascade.temporal_metrics.summary()

print('=== Temporal Security Metrics ===')
print(f'Total detections:          {temporal["total_detections"]}')
print(f'Early Intervention Rate:   {temporal["early_intervention_rate"]:.2%}')
print(f'Damage Prevented:          {temporal["damage_prevented_pct"]:.1f}%')
print(f'\nDetection Gap:')
gap = temporal['detection_gap']
print(f'  Mean:   {gap["mean"]:.1f} steps')
print(f'  Median: {gap["median"]:.1f} steps')
print(f'  Max:    {gap["max"]} steps')
print(f'\nDetections by layer:')
for layer, count in temporal['detections_by_layer'].items():
    print(f'  {layer}: {count}')

## 8. Per-Trajectory Timeline Analysis

In [ ]:
# Show detection timelines for a few attack trajectories
attack_detected = results_df[(results_df['is_attack']) & (results_df['detected'])]

print('=== Detection Timeline Examples ===')
for _, row in attack_detected.head(5).iterrows():
    traj = next(t for t in dataset if t['job_id'] == row['job_id'])
    attack_start = traj.get('attack_start_step', '?')
    print(f'\nJob: {row["job_id"]} ({row["label"]})')
    print(f'  Attack started at step: {attack_start}')
    print(f'  First detection at step: {row["first_detection_step"]} (Layer {row["first_detection_layer"]})')
    if isinstance(attack_start, int) and isinstance(row['first_detection_step'], int):
        gap = row['first_detection_step'] - attack_start
        print(f'  Detection gap: {gap} steps')

## Summary

Key evaluation findings:

1. **Cascade design works**: Layer 1 handles the majority of events at microsecond latency, only escalating ambiguous events to ML and LLM layers
2. **Weight exfiltration** is detected early through Layer 1 rules (production model access, denied egress)
3. **Temporal metrics** show early intervention -- most attacks are caught within a few steps of beginning
4. **Damage prevented** metric quantifies the value of early detection over post-hoc analysis
5. The 3-layer cascade balances detection accuracy against computational cost